# Enterprise Contract Guardian — GRPO Training

**Meta PyTorch OpenEnv Hackathon × Scaler School of Technology — Grand Finale**

This notebook trains a small open-weight model (Qwen2.5-1.5B by default) on the API Contract Validator environment using GRPO from TRL. The reward signal comes directly from the deployed environment, not from a static dataset — the model learns by interacting with the env on every training step.

**Hardware**: T4 GPU (15 GB VRAM) is enough.  Colab free tier or HF Jobs `--flavor t4-small` both work.

**Estimated runtime**: ~45 min for 200 steps on Qwen2.5-1.5B at LoRA r=16.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q openenv-core==0.2.2 trl unsloth wandb matplotlib datasets
!pip install -q --no-deps bitsandbytes triton xformers

## 2. Authenticate to HuggingFace and WandB

In [ ]:
import os
from huggingface_hub import login as hf_login
import wandb

# Either paste your tokens here, or `Runtime → Secrets` in Colab
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN') or 'hf_paste_yours_here'
os.environ['WANDB_API_KEY'] = os.getenv('WANDB_API_KEY') or 'paste_yours_here'

hf_login(token=os.environ['HF_TOKEN'])
wandb.login(key=os.environ['WANDB_API_KEY'])

## 3. Clone the project repo and start the env server

We start a local FastAPI server in the background. If you already have an HF Space deployed, set `ENV_URL` to the Space URL instead and skip the server start.

In [ ]:
!git clone https://github.com/kumarpushpam17-personal/Hackathon hack && cd hack/api_contract_validator && pip install -e .
%cd hack/api_contract_validator

# Start the env server in the background
import subprocess, time
server = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
time.sleep(8)
!curl -s http://localhost:7860/health

## 4. Run the baseline (untrained) to establish before-numbers

Writes `baseline_scores.json` at the repo root. This is one of the two files the judges grade the "Improvement in Rewards" 20% criterion against.

In [ ]:
%env API_BASE_URL=https://router.huggingface.co/v1
%env MODEL_NAME=Qwen/Qwen2.5-72B-Instruct
%env BASELINE_OUT=/content/hack/baseline_scores.json

!python training/baseline.py

## 5. Train with GRPO

The reward function rolls out one env step per generated completion and uses the env's grader as the reward. GRPO compares the `num_generations` completions per prompt and pushes the model toward the higher-reward ones.

In [ ]:
%env BASE_MODEL=unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
%env ENV_URL=http://localhost:7860
%env MAX_STEPS=200
%env NUM_GENERATIONS=4
%env WANDB_PROJECT=openenv-contract-guardian
%env WANDB_RUN=grpo-onsite
%env PUSH_TO_HUB=YOUR_USERNAME/api-contract-validator-grpo

!python training/train.py

## 6. Run inference with the trained adapter

Set `MODEL_NAME` to the adapter you just pushed, then re-run inference and write `trained_scores.json`.

In [ ]:
%env MODEL_NAME=YOUR_USERNAME/api-contract-validator-grpo
%env SCORES_OUT_PATH=/content/hack/trained_scores.json

!python inference.py

## 7. Generate the plots judges look at

Writes `results/reward_curve.png` and `results/before_after.png`. Commit these to the repo — they're embedded in the README results section.

In [ ]:
!python training/plot.py
from IPython.display import Image, display
display(Image('results/reward_curve.png'))
display(Image('results/before_after.png'))

## 8. Commit the artefacts back to your fork

From your laptop after the run is done:

```bash
git add baseline_scores.json trained_scores.json \
    api_contract_validator/results/reward_curve.png \
    api_contract_validator/results/before_after.png
git commit -m 'Add baseline, trained scores and reward plots'
git push
```